# 04 — Évaluation et preuves de fonctionnement

Cette évaluation sépare les scores de classification sur validation et les contrôles fonctionnels de traçabilité. Aucun score de pertinence ou de fidélité humaine n’est inventé.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Ouvrir depuis la racine ou notebooks/'
sys.path.insert(0, str(ROOT))
import json
import sqlite3
from contextlib import closing
import pandas as pd
REPORTS = ROOT / 'reports/demo'
DATABASE = ROOT / 'data/processed/demo.sqlite'
assert DATABASE.is_file(), 'Executer python -m scripts.prepare_demo depuis la racine'


In [2]:
metrics = json.loads((REPORTS / 'baseline_metrics.json').read_text(encoding='utf-8'))
pd.DataFrame([{'partition': split, 'effectif': score['rows'], 'macro_f1': score.get('macro_f1')}
              for split, score in metrics['splits'].items()])

,partition,effectif,macro_f1
0,validation,497,0.496604


In [3]:
pd.read_csv(REPORTS / 'confusion_validation.csv', index_col=0)

,Checking or savings account,Credit card,Credit reporting or other personal consumer reports,Debt collection,Debt or credit management,"Money transfer, virtual currency, or money service",Mortgage,"Payday loan, title loan, personal loan, or advance loan",Prepaid card,Student loan,Vehicle loan or lease
Checking or savings account,24,2,0,0,0,0,0,0,0,0,0
Credit card,1,16,2,0,0,1,0,0,0,0,0
Credit reporting or other personal consumer reports,0,6,292,31,0,0,0,0,0,7,0
Debt collection,1,3,15,41,0,0,1,0,0,1,0
Debt or credit management,0,0,1,1,1,0,0,0,0,0,0
"Money transfer, virtual currency, or money service",7,1,0,0,0,8,0,0,0,0,1
Mortgage,2,0,1,1,0,0,6,0,0,0,0
"Payday loan, title loan, personal loan, or advance loan",3,0,0,2,0,0,0,0,0,0,1
Prepaid card,2,1,0,0,0,1,0,0,0,0,0
Student loan,2,0,1,0,0,0,0,0,0,4,0


## Contrôles fonctionnels
Vérifier que chaque extrait provient du récit stocké et que ses métadonnées correspondent au corpus. Ces assertions évaluent la traçabilité du mode extractif, sans évaluer la compréhension ou la qualité d’une synthèse.

In [4]:
import joblib
from src.rag import ask
index = joblib.load(ROOT / 'artifacts/demo/retriever.joblib')
checks = []
with closing(sqlite3.connect(DATABASE)) as db:
    for question in ['credit report identity theft', 'mortgage payment escrow']:
        result = ask(index, question)
        assert result['sources']
        for source in result['sources']:
            row = db.execute('SELECT narrative, source_file, split FROM complaints WHERE complaint_id=?', (source['complaint_id'],)).fetchone()
            assert row and row[0].startswith(source['excerpt'])
            assert row[1] == source['source_file'] and row[2] == 'train'
        checks.append({'question': question, 'sources_verifiees': len(result['sources'])})
assert ask(index, 'quasar nebula astrophysics')['mode'] == 'abstention'
assert not metrics['test_evaluated']
pd.DataFrame(checks)

,question,sources_verifiees
0,credit report identity theft,3
1,mortgage payment escrow,3


In [5]:
manifest = json.loads((REPORTS / 'manifest.json').read_text(encoding='utf-8'))
{'scope': manifest['scope'], 'python': manifest['python'], 'packages': manifest['packages'], 'test_2026_opened': manifest['test_2026_opened']}

{'scope': 'Demonstration sur les premieres lignes de chaque export 2025, non representatif',
 'python': '3.11.9',
 'packages': {'pandas': '2.2.3', 'scikit-learn': '1.8.0', 'joblib': '1.5.3'},
 'test_2026_opened': False}

## Évaluation finale à réaliser
Constituer un jeu de questions indépendant avec sources pertinentes annotées avant la sélection du moteur. Rapporter Recall@3, Recall@5 et MRR, puis vérifier manuellement chaque affirmation et citation des synthèses générées. Ouvrir le test 2026 seulement après fixation des choix : `python -m src.classifier --evaluate-test`. Les résultats de démonstration ne remplacent pas cette évaluation finale.